In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 12.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.datasets as tvdata

from torchvision.transforms import transforms
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler

import matplotlib.pyplot as plt
import numpy as np

import math
from tqdm.auto import tqdm


from torchmetrics import Accuracy

In [3]:
"""
computer_vision
image_classification:
 - input_image -> 0 or 1 (binary classification)
 - input_image -> x {probability that the input image) belongs to one of the available classes or labels (multiclass classification)
object_detection:
 - input_image -> along with classification, draws boxes or some other representations around the objects
neural_style_transfer:
 - input_image + style_image -> returns input_image in the format or style of style_image

image_representation: images are generally converted into tensors in the shape of CxHxW; where C: color_channel, H: height, W: weight


convolutions: one of the fundamental building blocks of CNN networks
# Edge-detection fundamental technique in computer vision and image process that identifies abrupt pixel intensity changes that help in finding sharp discontinuations or boundaries
 which could therefore be used to outline the contours of an object, a person, or a car texture transition, or surface boundaries
 - vertical_edges: detects all the vertical lines in an image
 - horizontal_edges: detects all the horizontal lines in an image


An abstract overview/representation of how CNN could be comprehended:

- so, the greyscale image says if it has nxn pixels, then each pixel is normalized to say values that range
 from 0 to 9 including 0 and 9, where if the intensity number changes then it means that there is a line,
 so if the numbers change in the horizontal or pixels next to each other it means there is a change in
 pixel intensity between those two and the same thing for pixels that are above and below or
 on either side for any given pixel, and if say the grid has the same numbers it means that the pixel is just blank with the same intensity,
 if the values change seamlessly then no edge can be seen directly,
 but if the values change abruptly in any direction then there is an edge where it could be on any of the sides of the cells

- now to detect vertical edges in the above representation, one approach could be to construct a 3x3 filter with 1's in the first column, 0's in the second column, and -1's in the third column,
 when the image is convolved or if the convolution of this filter is applied to the image, a 3x3 filter is taken and applied to the image,
 where the element-wise product of the image where the filter is applied is taken with respective values in the filter and then the results are added to get a value,
 which is then repeated across the image where the image slides across the image

- if in this case, for instance, the first column of the 3x3 block of the image has all 5's the filter applied on that gives 15,
 where 5's of that column are multiplied by the first column of the filter,
 if the third or the rightmost column has all 1s then the filter applied gives 3
 and the middle column of that grid on the image is nullified as it is multiplied by 0,
 where now the filter applied emphasizes the changes in the intensity values, therefore giving a positive value
 if the values on both the right and left columns are the same in the image, then the result of applying the filter is 0, which shows that there is no significant change,
 however, the values in the second column aren't nullified, as if those values have significant values,
 then they are accounted for when the filter strides or moves in the next iteration


- in a similar way, the horizontal filter could also be applied to detect horizontal edges in an image

- if there is more than one channel in the image for example if they are RGB or some other representation
 which could have n values that could be combined to represent an image, then applying each of those could yield additional information which
 can be layered in the form of volume to represent that image, for example in RGB, a red filter could be applied to detect changes in the red color in the image,
 where instead of horizontal or vertical edges, the focus is on shifts in the red colors of the image,
 when the filter is applied then the resultant of that convolution operation respectively for green and blue can be
 put together which can be conceptualized as a volume rather than a flat image, it could then be used to detect edges or contours or any shapes in the given image.

- frameworks such as tensorflow pytorch or keras are commonly used to perform convolution operations which offer simple methods that hide the complexities of the convolution operations.
 conv2d operation in PyTorch for example takes in_channels as the number of channels that are used to represent the input image,
 where if there are n channels and the size of the image is x, x for height and width respectively,
 then conv2d can be used to increase the number of channels that are either greater than or less than n,
 which could help extract features or representations of patterns even if they are minute or complicated.
 the filters that conv2d applied are learnable, and are adjusted during the training process,
 where the filter's parameters are adjusted to capture the edge patterns or features in a more effective way
 where the filter's parameters are adjusted to capture the edge patterns or features in a more effective way
 (note: in general, images are resized which is not a compulsion, but are done for efficient computing and simplicity,
 as the images move through the convolution if not most of the details or features are compensated via learning,
 it may not be ideal for every situation, but in general, if there are large datasets or if data is augmented for training,
 then it would be ideally okay to resize but again this would be based on the choice)

- filters or kernels vary in values in cells or size of the filter

- in deep learning, the values in filters are learnable parameters, generally referred to as weights which are learned using backpropagation
 where such filter or kernel when convolved with images gives proper values or representations of edges,
 where it could find edges that may have any orientation or any features or patterns

- size or dimension of the output after convolution: (nxn) grid, (fxf) filter = (n-f+1) x (n-f+1)


# convolutions_over_volumes: convolution where there are more than 1 color channel or
 any other format that is used to describe an image rather than just a flat 2d representation
- for instance, if an image has nxn dimension with 3 color channels (RGB), to detect edges or features,
 then the image could be represented as (3, n, n), where 3 represents the number of color channels and n,n could represent height and width respectively

- to perform a convolution operation on (c, n, n) image, it is convolved with (c, f, f ) filter/kernel,    where c in both the image and filter represents the number of channels
while the n,n in the image and f,f in the filter represent the dimension or size of the image/filter respectively  the resultant output of this would be (n-f+1)x(n-f+1) image
- for instance, if the image and filter respectively have 3 channels, the convolution operation multiplies each of the layers respectively,
 where the first channel of the filter convolves with the first channel of the image and repeats the same for the second and third channels,
 which are then added element-wise similar to the one-channel convolution operation and are traversed across the image
 the resultant image is thus (n-f+1)x(n-f+1)

- the same operation could be performed more than once which could be either to detect additional features or patterns or edges or any such thing,
 where in doing so, applying multiple such filters would also add depth or volume to the output of the convolution operation,

- thus, if one filter is applied then it gives an image say (n-f+1) x (n-f+1), then adds say d convolution operations,
 it would result in (n-f+1) x (n-f+1) x d volume that represents the output of convolutions of filter/kernels operated on the input image

- the size of the color_channels in the image must be equal to the depth of the filter applied,
 where if the image has size (c, n, n) then each of the convolution filter/kernel must also have the size of (c, f, f)

- if the image is of size (3, 6, 6) where it represents color_channel, height, and width respectively,
 then combining it with the (3x3x3) filter gives an image of size 4x4, and convolving it again with another filter of size (3x3x3) gives another image of size (4x4),
 where stacking them on top of each other would give a 4x4x2 volume that represents the output of the convolution operations applied to the input image

- For example: if X is an input image, k1 and k2 are kernels/filters where k1 and k2 are of the same size,
 convolving X and k1 and k2 gives outputs o1 and o2 respectively, then each of those is added with respective bias and an activation function is applied on them independently
 which gives the respective images, and then both the outputs after activations applied are stacked together

# padding: extra cells around the grid that represent the image
- if the filter or kernel size is f, then the padding must add f - 1 cells around the edges to the output grid to have the same dimension as the input
 where in doing so it prevents shrinking the outputs or reducing the size as the layers grow

 - it also helps in better emphasis on the cells on the corners which are considered lesser times than those in the center.
 whereby adding padding the emphasis or using more than once, which thus helps in preventing any
 loss of information or features from the edges
-
 network may still learn to represent the images even without applying padding as it adjusts the weights to better represent the values,
 adding padding is generally known to reduce the complexity that the parameters may otherwise have,
 which may not be the case in every situation, however, there are other techniques that are used to effectively reduce the size without distorting or diminshing the effectiveness of the network

- if p is padding, p=1, output = (n+2p-f+1) x (n+2p-f+1)

- valid convolutions: no-padding
- same convolution: padding such that the output size is the same as the input size

- same concolution: (nxn) grid, (fxf) filter:
 - for the output image to be the same, the padding value 2p should be of size f - 1, p = (f-1)/2, where f is odd
 output = (n+2p-f+1) x (n+2p-f+1)  => n + 2 (f -1)/2 - f + 1 = n => the output thus is next
 - if f is even then generally asymmetric padding is applied where thee might be extra padding on either side

# stridden convolutions: filter/kernel skips some cells as it traverses through the image
- if the stride value is 2, application of filter or the convolution operation skips 2 cells as it traverses
- if an image is convolved with fxf filter with padding p, stride s:
 output size = ((n+ 2p -f )/s) + 1 x  ((n+ 2p -f )/s) + 1,
 if  ((n+ 2p -f )/s) + 1 is not an integer, the floor of the number is taken into consideration


# pooling: downsampling operation after convolution operations which reduces the spatial dimensions (height, width)
 which decreases the number of parameters therefore effectively reducing computational cost and complexity, and
 helping the network to learn more effectively to generalize and also reducing issues of overfitting

- similar to convolutions, a pooling filter traverses through the image and outputs the value based on the applied computation
 - for instance, in MaxPooling, a 2x2 filter with a stride of 2 can be used to traverse through the image,
 where it takes the maximum value in each of the 2x2 grids of the image and outputs the max value in that grid. And reducing the image size by half
- there are no learning parameters in pooling in general
- pooling reduces the size of the image but in general, does not reduce the depth of the input,
 where if there are c number of channels in the input, then max-pooling is applied to each of the c channels independently
- The output size of max-pooling is similar to that of applying filter./kernel for convolution operation
 = ((n+ 2p -f )/s) + 1 x  ((n+ 2p -f )/s) + 1, where if the value is not an integer the floor value of the number is taken as a resultant size
-Average pooling is also another technique applied, where it takes the average number


# fully_connected_layer: connects every neuron from a layer to every neuron in the following layer
 - if  there is an input of size (c, n, n) then the layer is flattened to c * n * n and then fully connected to the next layer,
 where every neuron from this flattened layer is connected to every neuron in the following layer,
 the neurons in this following could be lesser or greater than the flattened layer

- if there are x number of classes or target labels, then the fully connected layer from the previous step is connected to this layer with x number of neurons,
 where the outputs of this layer are logits, on which a softmax is applied to get the probabilities of which class the input could belong to

# common layers in CNN (convolutional neural network):  convolution, pooling, fully_connected
"""



"\ncomputer_vision\nimage_classification:\n - input_image -> 0 or 1 (binary classification)\n - input_image -> x {probability that the input image) belongs to one of the available classes or labels (multiclass classification)\nobject_detection:\n - input_image -> along with classification, draws boxes or some other representations around the objects\nneural_style_transfer:\n - input_image + style_image -> returns input_image in the format or style of style_image\n\nimage_representation: images are generally converted into tensors in the shape of CxHxW; where C: color_channel, H: height, W: weight\n\n\nconvolutions: one of the fundamental building blocks of CNN networks\n# Edge-detection fundamental technique in computer vision and image process that identifies abrupt pixel intensity changes that help in finding sharp discontinuations or boundaries\n which could therefore be used to outline the contours of an object, a person, or a car texture transition, or surface boundaries\n - verti

In [4]:
import os
# retrieve the number of cpu's available for use

RANDOM_SEED = 32 # to ensure that the model can hepl results more reproducible
NUM_CPUS = os.cpu_count()
BATCH_SIZE = 32
MODEL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
from math import floor
class ModularCNN(nn.Module):
    """a multiclass convolutional neural network model that takes input image and gives probabilities of classes"""
    def __init__(self, input_channels=3, target_labels=10, image_size=64):
        super(ExpCNN, self).__init__()
        """
        Args:
            input_channels (int): number of color_channels or any such representation of the input image, ex: if it is an RGB image, then the color_channels=3
            target_labels (int): number of classes or target variables in the data
            image_size: size of the input image
        """

        """
        the input image is of size (batch_size, color_channels, height, width)
        """
        # input: batch_size: 64, color_channels: 3, height: 64, width: 64
        # padding of size 1 is added to the convolutional operations to keep the image size the same
        self.block_1 = nn.Sequential(
            # input (64, 3, 64, 64) -> output (64, 16, 64, 64)
            nn.Conv2d(in_channels=input_channels, out_channels=16, kernel_size=3, stride=1, padding=1), # increases the channels or depth from 3 to 16
            nn.BatchNorm2d(16), # normalizes all the values in each of the 16 channels in the layer
            nn.ReLU(), # activation layer

            # input(64, 16, 64, 64) -> output (64, 64, 64, 64)
            nn.Conv2d(in_channels=16, out_channels=64, kernel_size=3, stride=1, padding=1), # increases the channels or depth from 16 to 64
            nn.BatchNorm2d(64), # normalizes all the values in each of the 64 channels
            nn.ReLU(),

            # input(64, 64, 64, 64) -> output (64, 64, 32, 32)
            nn.MaxPool2d(kernel_size=2) # reduces the size of image but does not effect the depth
        )


        # input (output of previous block): batch_size: 64, color_channels: 64, height: 32, width: 32
        self.block_2 = nn.Sequential(
            #input (64, 64, 32, 32) -> output(64, 128, 32, 32)
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1), # increases the channels/depth from 64 to 128
            nn.BatchNorm2d(128), # normalizes all the values in each of the 128 channels
            nn.ReLU(),

            # input(64, 128, 32, 32) -> output(64, 128, 32, 32)
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1, stride=1), # channels/depth size remains the same (128)
            nn.BatchNorm2d(128), # normalizes all the values in each of the 128 channels
            nn.ReLU(),

            # input(64, 128, 32, 32) -> output (64, 128, 16, 16)
            nn.MaxPool2d(kernel_size=2) # reduces the size of the image
        )

        # input (output of previous block): batch_size: 64, color_channels: 128, height: 16, width: 16
        self.block_3 = nn.Sequential(
            #input  (64, 128, 16, 16) -> output(64, 256, 16, 16)
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1), # increases the channels/depth from 128 to 256
            nn.BatchNorm2d(256), # normalizes all the values in each of the 256 channels
            nn.ReLU(),

            # input(64, 256, 16, 16) -> output(64, 256, 16, 16)
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1, stride=1), # channels/depth size remains the same (256)
            nn.BatchNorm2d(256), # normalizes all the values in each of the 256 channels
            nn.ReLU(),

            # input(64, 256, 16, 16) -> output (64, 256, 8, 8)
            nn.MaxPool2d(kernel_size=2) # reduces the size of the image
        )

        # fully connected layer that converts convolutional blocks into linear layers and then to length of class labels
        # input (64, 256, 8, 8) -> (64, 256 * 8 * 8) -> (64, 10)

        # floor(image_size/8) is explicity for this code as here maxpool reduces the image size to 1/8th and by standard convention the floor value of non-integer number is taken
        #self.linear_hidden_features = 256 * (image_size//8) * (image_size//8)



        def init_linear_features():
            # dummy inputs to generate number of outputs for the in_features for linear layer that are the outputs after flattening the block_3
            dummy_input = torch.randn(1, input_channels, image_size, image_size) # generates a dummy input tensor with shape (batch_size=1, in_channels=3, height=image_size, width=image_size)

            with torch.inference_mode():
                #input_shape(1, 3, 64, 64) -> output_shape(1, channels, height, width) from convolution operations
                dummy_output = self.block_3(self.block_2(self.block_1(dummy_input))) # passes the dummy input throuhg the three blocks

            # return the number by dividing the total number of elements by dividing it with batch_size
            # if the input is (1, 3, 64, 64), after applying the convolution and max pool in this code, it gives (1, 256, 8, 8)
            # total number of elements in dummy_output.numel() = 1 * 256 * 8 * 8 then dividing it by dummy_output.shape[0] here it is 1
            # but again if it it some other number, then dividing it by the batch size gives the total number of elements in channels * width * height
            return dummy_output.numel() // dummy_output.shape[0]


        # calculate flattened features
        self.linear_hidden_features = init_linear_features()
        self.classification_block = nn.Sequential(

            # input (64, 256, 8, 8) -> output (64, 256 * 8 * 8)
            nn.Flatten(), # starts from dim 1, since dim 0 is the batch_number, here it takes color_channels, height, and width and transforms into a flattened vector

            # input(64, 256 * 8 * 8) -> output (64, 1024)
            nn.Linear(in_features=self.linear_hidden_features, out_features=1024),
            nn.ReLU(),

            nn.Dropout(0.5), # dropouts nodes with a probability of 0.5 that could help prevent overfitting or memoization
            # input(64, 1024) -> output(10), where the given target label is the number of classes
            nn.Linear(in_features=1024, out_features=target_labels)
        )


    # forward propagation
    def forward(self, x):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = self.classification_block(x)
        return x

In [6]:
# converting the above code block to modular

def build_conv_block(in_channels, out_channels):
    """
    Args:
        in_channels: number of channels in the input
        out_channels: number of channels in the output

    if the input is of size (batch_size, in_channels, floor(image_size/2), floor(image_size/2)):
     this method returns (batch_size, out_channels, floor(image_size/2), floor(image_size/2))
    """
    return nn.Sequential(
        nn.Conv2d(in_channels,  out_channels, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )

class ModularCNN(nn.Module):
    def __init__(self, input_channels=1, output_channels=10, image_size=28, num_blocks=1):

        super(ModularCNN, self).__init__()

        self.num_blocks = num_blocks
        self.block_1 = build_conv_block(input_channels, 64)



In [7]:
# fashion mnist dataset to check the model
torch.manual_seed(RANDOM_SEED)
"""
each data is of size 1x28x28, the images are converted to tensors
dataloader is of size 32, so each batchsample of dataloader would be represented as  (64, 1, 28, 28)
"""
fashionmnist_dataset_train = tvdata.FashionMNIST(root='/data/fashionmnist', train=True, transform=transforms.ToTensor(), download=True) # train dataset
fashionmnist_dataset_test = tvdata.FashionMNIST(root='/data/fashionmnist', train=False, transform=transforms.ToTensor(), download=True) # test dataset

# use randomsampler to split training data into test and validation
valid_data_size = 0.1 # percentage of training data that should be used for validation
len_train_dataset = len(fashionmnist_dataset_train)
indicies = list(range(len_train_dataset))
split = int(np.floor(valid_data_size * len_train_dataset))
train_idx, valid_idx = indicies[split:], indicies[:split]

train_sampler = SubsetRandomSampler(train_idx)
valid_sampler = SubsetRandomSampler(valid_idx)

fmnist_train_dataloader = DataLoader(fashionmnist_dataset_train, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_CPUS) # train dataloader
fmnist_valid_dataloader = DataLoader(fashionmnist_dataset_train, batch_size=BATCH_SIZE, sampler=valid_sampler, num_workers=NUM_CPUS) # valid dataloader
fmnist_test_dataloader = DataLoader(dataset=fashionmnist_dataset_test, batch_size=BATCH_SIZE) # test dataloader

fashion_mnist_class = fashionmnist_dataset_train.classes
fashion_mnist_class_idx = fashionmnist_dataset_train.class_to_idx
image, label = fashionmnist_dataset_train[0]

print(f'len_train: {len(fashionmnist_dataset_train)} | len_test: {len(fashionmnist_dataset_test)}')
print(f'fashion_mnist_classes: {fashion_mnist_class}')
print(f'fashion_mnist_classes_idx: {fashion_mnist_class_idx}')
print(f'input_image_shape: {image.shape}')

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 274kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.06MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.9MB/s]

len_train: 60000 | len_test: 10000
fashion_mnist_classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
fashion_mnist_classes_idx: {'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}
input_image_shape: torch.Size([1, 28, 28])


In [8]:
model_cnn_0 = ExpCNN(input_channels=1, image_size=28).to(MODEL_DEVICE)

# activation function: adam is used here, since it is standard for cnn
optimizer = optim.Adam(params=model_cnn_0.parameters())

# cross entropy loss is used, as per convention
criterion = nn.CrossEntropyLoss()

# number of epochs or training loops
NUM_EPOCHS = 10


accuracy_metrics = Accuracy(task="multiclass", num_classes=10)

NameError: name 'ExpCNN' is not defined

In [ ]:
# a simple function to keep a track of the training time
from timeit import default_timer

def train_time(start, end, device):
    total_time = end - start
    return f'train_time on {device}: {total_time}'

In [ ]:
# training the model and  using tqdm to display progress bar

torch.manual_seed(RANDOM_SEED)
start_time = default_timer()

for epoch in tqdm(range(NUM_EPOCHS)):
    print(f"Epoch: {epoch}\n---")

    accuracy_metrics.reset()

    # metrics for training
    train_loss_list = [] # append the training loss per batch, keeps track of individual losses, it could be summed up and divided to get the averge training loss per batch
    eval_loss_list = [] # similar to training loss, but used in evaluation loss which is done in evaluation method
    eval_accuracy_list = [] # evaluate predictions against the truth labels and append to the list
    for batch, (images, labels) in enumerate(fmnist_train_dataloader): # traverses or iterates over each batch

        images, labels = images.to(MODEL_DEVICE), labels.to(MODEL_DEVICE)

        model_cnn_0.train() # set the model to train mode
        optimizer.zero_grad() # reset the optimizers gradients to prevent from accumulation

        train_logits = model_cnn_0(images) # predict the outputs, where the model outputs logits
        train_loss = criterion(train_logits, labels) # calculate the loss between the predicted logits and the actual truth values of target

        train_loss_list.append(train_loss.item()) # append the loss to the list

        train_loss.backward() # back propagation
        optimizer.step() # updates weights and respective learnable parameters


    # evaluate the model's predictions
    model_cnn_0.eval() # set the model to evaluation mode
    for eval_batch, (eval_images, eval_labels) in enumerate(fmnist_valid_dataloader): # traverses or iterates over each batch
        eval_images, eval_labels = eval_images.to(MODEL_DEVICE), eval_labels.to(MODEL_DEVICE) # move the devices to MODEL DEVICE
        with torch.inference_mode(): # disables gradient descent

            eval_logits = model_cnn_0(eval_images) # predict logits on validation data
            eval_loss = criterion(eval_logits, eval_labels) # calculates loss on predicted logits
            eval_loss_list.append(eval_loss.item())
            eval_predictions = torch.argmax(torch.softmax(eval_logits, dim=1), dim=1)
            eval_accuracy = accuracy_metrics(eval_predictions, eval_labels).item()
            eval_accuracy_list.append(eval_accuracy)

    # average out and printing losses and accuracy
    avg_train_loss = sum(train_loss_list) / len(train_loss_list)
    avg_eval_loss = sum(eval_loss_list) / len(eval_loss_list)
    avg_eval_accuracy = sum(eval_accuracy_list) / len(eval_accuracy_list)
    print(f'avg_train_loss: {avg_train_loss:.4f} | avg_validation_oss: {avg_eval_loss:.4f} | validation_accuracy: {avg_eval_accuracy:.2f}%')

end_time = default_timer()




In [ ]:


IMAGE_SIZE = 64 # size that the raw images are adjusted to before being transformed
BATCH_SIZE=64 # batch size of the images used in dataloaders
COLOR_CHANNELS=3 # number of colors channels of input images used in the model

# normalized mean and standard deviation values that are used in transforming data, set per CIFAR10 datset
NORMALIZED_MEAN=[0.4914, 0.4822, 0.4465]
NORMALIZED_STD = [0.2023, 0.1994, 0.2010]

def cifar10data(
                datadir='/data/cifar10/',
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                random_seed = RANDOM_SEED,
                test=False,
                valid_size=0.1,
                normalized_mean = NORMALIZED_MEAN,
                normalized_std = NORMALIZED_STD,
                num_cpus = NUM_CPUS):

    # manual seed to ensure that the data can be reproducible
    torch.manual_seed(random_seed)
    """
    loads cifar10 dataset and returns dataloaders
    Args:
        - datadir (str): root directory or the folder to where the data must exist
        - image_size (int): size of the image, if the input is 64,  then the resized image would be of size (64, 64)
        - train (bool): if True, will use the train split of the dataset
        - valid_size(float): size of the validation dataset from the training dataset, if the valid_size is 0.1, then the valid_dataset size would be 10% of the training dataset
        - batch_size= size of batch that DataLoader process  (number of samples per batch)
        - valid_size: size used to split training data into train and valid dataloaders
        - normalized_mean: values used to normalize the mean of tensor representation of data
        - normalized_std: values used to normalized the standard deviation of tensor representation of data
    Return:
        - if train is set to false, will return test DataLoader
        - if train is set to true, splits cifar10 dataset into train and test data and returns respective dataloaders along with class labels
    """

    # resizes the data to the given image size and transforms them to tensors
    tranform_data = transforms.Compose([
        transforms.Resize((image_size, image_size)), # resizes the image to 64x64
        transforms.ToTensor(), # converts the data to tensors
        transforms.Normalize(mean=normalized_mean, std=normalized_std)
    ])

    # if the train value is set to false, then this will download and build test dataloader
    if test:
        test_dataset = tvdata.CIFAR10(root=datadir, train=False, transform=tranform_data, download=True)
        dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size, num_workers=num_cpus)
        return dataloader
    else:
        train_dataset = tvdata.CIFAR10(root=datadir, train=True,transform=tranform_data, download=True)
        len_train_dataset = len(train_dataset)
        indicies = list(range(len_train_dataset))
        split = int(np.floor(valid_size * len_train_dataset))
        train_idx, valid_idx = indicies[split:], indicies[:split]

        train_sampler = SubsetRandomSampler(train_idx)
        valid_sampler = SubsetRandomSampler(valid_idx)

        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=num_cpus)
        valid_dataloader = DataLoader(train_dataset, batch_size=batch_size, sampler=valid_sampler, num_workers=num_cpus)
        cifar10_classes = train_dataset.classes
        return train_dataloader, valid_dataloader, cifar10_classes

"""

In [ ]:
train_cifar10_data, valid_cifar_data, cifar10_classes = cifar10data() # load trianing data and return train and valid dataloaders
test_cifar10_data = cifar10data(test=True) # load testing data and return test dataloader

train_cifar_iter = iter(train_cifar10_data)

images, labels = next(train_cifar_iter)

print(f'image_shape (batch_size, color_channel, height, width): {images.shape} | len_class_labels: {len(cifar10_classes)}')
print(f'class_names: {cifar10_classes}')